In [3]:
import pandas as pd
import numpy as np
import math


CSV_FILE = "./pisa_data.csv"

data = pd.read_csv(CSV_FILE)


responses = data.pivot_table(index='student_id', columns='item_id', values='response')

final_scores = data.groupby('student_id')['final_score'].first()

students_df = responses.copy()
students_df['final_score'] = final_scores

# Discretize final_score into 8 intervals
students_df['ability_level'] = pd.qcut(students_df['final_score'], q=8, labels=False)

question_features = list(responses.columns)

# Compute median response time for each question
median_response_time = data.groupby('item_id')['response_time'].median().to_dict()


def gini_index(labels):
    """
    Compute the Gini index for a set of discrete labels.
    Gini = 1 - sum(p_k^2)
    where p_k is the probability of samples belonging to class k.
    """
    if len(labels) == 0:
        return 0
    counts = np.bincount(labels)
    probabilities = counts / len(labels)
    return 1.0 - np.sum(probabilities**2)

def gini_gain(data_subset, feature):
    """
    Calculate the Gini gain of splitting data_subset on the given feature.
    The label is 'ability_level'.
    """
    # current gini
    current_gini = gini_index(data_subset['ability_level'].values)

    # split into two subsets based on the feature's value (0 or 1)
    left = data_subset[data_subset[feature] == 0]
    right = data_subset[data_subset[feature] == 1]

    # if the split is empty, no gain
    if len(left) == 0 or len(right) == 0:
        return 0

    # weighted average of gini in children
    p_left = len(left) / len(data_subset)
    p_right = len(right) / len(data_subset)
    children_gini = p_left * gini_index(left['ability_level'].values) + p_right * gini_index(right['ability_level'].values)

    return current_gini - children_gini

class Node:
    def __init__(self, question=None, left=None, right=None,
                 prediction=None, depth=0, time_used=0):
        """
        A node in the decision tree.
         - question: the item_id used to split (None if leaf)
         - left: branch for response == 0
         - right: branch for response == 1
         - prediction: the predicted ability level (leaf only)
         - depth: current depth in the tree
         - time_used: cumulative estimated time up to this node
        """
        self.question = question
        self.left = left
        self.right = right
        self.prediction = prediction
        self.depth = depth
        self.time_used = time_used

    def is_leaf(self):
        return self.prediction is not None

def build_tree(data_subset,
               available_features,
               current_depth,
               current_time,
               max_depth=15,
               time_limit=900):
    """
    Build the decision tree recursively.

    - data_subset: subset of students_df (rows) that reach this node
    - available_features: list of question features not yet used along this branch
    - current_depth: current depth
    - current_time: cumulative estimated time so far
    - max_depth: stop if we exceed this depth
    - time_limit: stop if we exceed this total time
    """
    if len(data_subset['ability_level'].unique()) == 1:
        return Node(
            prediction=data_subset['ability_level'].iloc[0],
            depth=current_depth,
            time_used=current_time
        )

    if (current_depth >= max_depth
        or len(available_features) == 0
        or current_time >= time_limit):
        majority_class = data_subset['ability_level'].mode()[0]
        return Node(
            prediction=majority_class,
            depth=current_depth,
            time_used=current_time
        )

    # choose the feature (question) that yields the highest gain and doesn't exceed time constraints
    best_feature = None
    best_gain = -1
    best_feature_time = 0

    for feature in available_features:
        est_time = median_response_time.get(feature, 0)
        # if adding this question's median time would exceed time_limit, skip
        if current_time + est_time > time_limit:
            continue

        gain = gini_gain(data_subset, feature)
        if gain > best_gain:
            best_gain = gain
            best_feature = feature
            best_feature_time = est_time

    # if no valid feature found (time constraint or zero gain), create leaf
    if best_feature is None or best_gain <= 0:
        majority_class = data_subset['ability_level'].mode()[0]
        return Node(
            prediction=majority_class,
            depth=current_depth,
            time_used=current_time
        )

    node = Node(
        question=best_feature,
        depth=current_depth,
        time_used=current_time + best_feature_time
    )

    # split the data based on correctness
    left_subset = data_subset[data_subset[best_feature] == 0]
    right_subset = data_subset[data_subset[best_feature] == 1]

    # remove used feature from the list
    remaining_features = [f for f in available_features if f != best_feature]

    # recursively build subtrees
    node.left = build_tree(
        left_subset,
        remaining_features,
        current_depth + 1,
        current_time + best_feature_time,
        max_depth,
        time_limit
    )
    node.right = build_tree(
        right_subset,
        remaining_features,
        current_depth + 1,
        current_time + best_feature_time,
        max_depth,
        time_limit
    )

    return node


decision_tree = build_tree(
    students_df,
    question_features,
    current_depth=0,
    current_time=0,
    max_depth=15,
    time_limit=900
)

def print_tree(node, indent=""):
    """
    Recursively print the tree structure for visualization.
    """
    if node.is_leaf():
        print(f"{indent}Leaf: Predict ability level = {node.prediction}, Time Used = {node.time_used:.2f}s")
    else:
        print(f"{indent}Question (item_id): {node.question}, Depth = {node.depth}, Time Used = {node.time_used:.2f}s")
        print(indent + "  [Response = 0] ->")
        print_tree(node.left, indent + "    ")
        print(indent + "  [Response = 1] ->")
        print_tree(node.right, indent + "    ")

print_tree(decision_tree)

Question (item_id): CG122Q03, Depth = 0, Time Used = 65.05s
  [Response = 0] ->
    Question (item_id): CS643Q01, Depth = 1, Time Used = 129.81s
      [Response = 0] ->
        Question (item_id): CS646Q01, Depth = 2, Time Used = 184.11s
          [Response = 0] ->
            Question (item_id): CS603Q01, Depth = 3, Time Used = 244.44s
              [Response = 0] ->
                Question (item_id): CG126Q01, Depth = 4, Time Used = 298.71s
                  [Response = 0] ->
                    Question (item_id): CM954Q01, Depth = 5, Time Used = 407.22s
                      [Response = 0] ->
                        Question (item_id): CS415Q02, Depth = 6, Time Used = 437.69s
                          [Response = 0] ->
                            Question (item_id): CS601Q01, Depth = 7, Time Used = 554.31s
                              [Response = 0] ->
                                Question (item_id): CS413Q06, Depth = 8, Time Used = 626.83s
                                  [R